In [ ]:
# Cell 1 — Mount Google Drive and check GPU

from google.colab import drive
drive.mount("/content/drive")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )

In [ ]:
# Cell 2 — Install required packages

!pip -q install -U transformers accelerate datasets jiwer soundfile tqdm
!pip -q install "pandas==2.2.3"

In [ ]:
# Cell 3 — Load the Tarifit tokenizer

from transformers import Wav2Vec2CTCTokenizer

VOCAB_PATH = (
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/"
    "data/processed/mms_tokenizer_v1_1/vocab.json"
)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit vocabulary size:", len(tokenizer))
print("PAD token ID:", tokenizer.pad_token_id)

In [ ]:
# Cell 4 — Load the fixed Tarifit train and validation datasets

from datasets import load_from_disk
from pathlib import Path

CORPUS_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/"
    "data/processed/mms_corpus_v1_1"
)

train_ds = load_from_disk(str(CORPUS_ROOT / "train"))
val_ds = load_from_disk(str(CORPUS_ROOT / "validation"))

print("Training examples:", len(train_ds))
print("Original validation examples:", len(val_ds))

print("\nTraining dataset:")
print(train_ds)

print("\nValidation dataset:")
print(val_ds)

In [ ]:
# Cell 5 — Create the cleaned 128-example validation set

BAD_VAL_INDICES = {111, 118, 126, 127, 130}

valid_val_indices = [
    i
    for i in range(len(val_ds))
    if i not in BAD_VAL_INDICES
]

val_clean_ds = val_ds.select(valid_val_indices)

print("Original validation:", len(val_ds))
print("Removed corrupted examples:", len(BAD_VAL_INDICES))
print("Clean validation:", len(val_clean_ds))

In [ ]:
# Cell 6 — Create the Fadhma feature extractor with the Tarifit tokenizer

from transformers import AutoFeatureExtractor, Wav2Vec2Processor

FADHMA_ID = "agbalu/Fadhma-300M"

feature_extractor = AutoFeatureExtractor.from_pretrained(FADHMA_ID)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Sampling rate:", feature_extractor.sampling_rate)
print("Audio normalization:", feature_extractor.do_normalize)
print("Tarifit output vocabulary:", len(tokenizer))

In [ ]:
# Cell 7 — Load Fadhma and replace the Kabyle CTC head with a Tarifit CTC head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    FADHMA_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

# Freeze the convolutional feature encoder
model.freeze_feature_encoder()

# Reduce GPU memory usage
model.gradient_checkpointing_enable()

model.config.mask_time_prob = 0.05
model.config.layerdrop = 0.0

model = model.to(device)

print("Model loaded.")
print("Output vocabulary:", model.config.vocab_size)
print("CTC head:", model.lm_head)

total_params = model.num_parameters()

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

In [ ]:
# Cell 8 — Compare the Fadhma and Tarifit vocabularies

from transformers import AutoProcessor

fadhma_original_processor = AutoProcessor.from_pretrained(
    "agbalu/Fadhma-300M"
)

fadhma_vocab = fadhma_original_processor.tokenizer.get_vocab()
tarifit_vocab = tokenizer.get_vocab()

fadhma_tokens = set(fadhma_vocab.keys())
tarifit_tokens = set(tarifit_vocab.keys())

special_tokens = {
    "[PAD]",
    "[UNK]",
    "|"
}

fadhma_letters = fadhma_tokens - special_tokens
tarifit_letters = tarifit_tokens - special_tokens

print("Fadhma tokens:")
print(sorted(fadhma_letters))

print("\nTarifit tokens:")
print(sorted(tarifit_letters))

print("\nShared:")
print(sorted(fadhma_letters & tarifit_letters))

print("\nOnly in Fadhma:")
print(sorted(fadhma_letters - tarifit_letters))

print("\nOnly in Tarifit:")
print(sorted(tarifit_letters - fadhma_letters))

print(
    "\nNumber shared:",
    len(fadhma_letters & tarifit_letters)
)

stop here and check the kabyle head feasability

In [ ]:
# Cell 9 — Verify CTC feasibility for the training and cleaned validation sets

import torch

def minimum_ctc_frames(labels):
    repeated_adjacent_tokens = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )

    return len(labels) + repeated_adjacent_tokens


def check_ctc_feasibility(ds, name):
    invalid = []

    for i, example in enumerate(ds):

        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(example["input_length"])
            ).item()
        )

        minimum_frames = minimum_ctc_frames(
            example["labels"]
        )

        if minimum_frames > output_frames:
            invalid.append(i)

    print(f"{name}:")
    print("  Total:", len(ds))
    print("  Valid:", len(ds) - len(invalid))
    print("  Invalid:", len(invalid))

    return invalid


bad_train = check_ctc_feasibility(
    train_ds,
    "TRAIN"
)

bad_validation = check_ctc_feasibility(
    val_clean_ds,
    "CLEAN VALIDATION"
)

In [ ]:
# Cell 10 — Create the CTC dynamic-padding data collator

from dataclasses import dataclass
from typing import Union
import torch


@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):

        input_features = [
            {"input_values": x["input_values"]}
            for x in features
        ]

        label_features = [
            {"input_ids": x["labels"]}
            for x in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

test_batch = data_collator([
    train_ds[0],
    train_ds[1]
])

print("Input shape:", test_batch["input_values"].shape)
print("Labels shape:", test_batch["labels"].shape)

In [ ]:
# Cell 11 — Test Fadhma-Tarifit forward and backward pass on the longest training segment

import torch

longest_idx = max(
    range(len(train_ds)),
    key=lambda i: train_ds[i]["input_length"]
)

example = train_ds[longest_idx]

print("Training index:", longest_idx)
print(
    "Duration:",
    round(example["input_length"] / 16000, 2),
    "seconds"
)

batch = data_collator([example])

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

model.train()
model.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    outputs = model(**batch)
    loss = outputs.loss

print("Loss:", loss.item())

loss.backward()

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print("✓ Forward and backward pass successful")

# No optimizer update is performed
model.zero_grad(set_to_none=True)